In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build the vocabulary of characters and mappings
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the data set
block_size = 3 # context length: how many chars are considered when predicting the next char

def build_dataset(words):
    X, Y = [],[]

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context) # stores the context for each char
            Y.append(ix) # stores the char
            context = context[1:] + [ix] # crop and append the new char to context

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1]) # Training set (80%)
Xdev, Ydev = build_dataset(words[n1:n2]) # Dev set (10%)
Xte, Yte = build_dataset(words[n2:]) # Test set (10%)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
# utility function to compare manual gradients to PyTorch gradients
def cmp(s, dt, t): # dt = our calc, t.grad = pytorch calc
    ex = torch.all(dt == t.grad).item() # boolean value of whether the two are exactly equal
    app = torch.allclose(dt, t.grad) # check if they're approximately equal due to floating points
    maxdiff = (dt - t.grad).abs().max().item() # maximum difference
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [7]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1 # biases are small nums (not 0) to mask incorrect gradient implementation -> if 0, simpler gradient expression than usual
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [8]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [25]:
# forward pass -> "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the chars into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate for multiplication

# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation

# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True) # mean pre-activation of hidden layer
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note Bessel's correction (dividing by n-1 = unbiased, divide by n = biased)
                                                # Batchnorm1d uses biased version for training and unbiased for test -> fine for large datasets but kind of a bug
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv # normalized variance
hpreact = bngain * bnraw + bnbias # batch norm layer pre-activation

# Non-linearity
h = torch.tanh(hpreact) # hidden layer

# Linear layer 2
logits = h @ W2 + b2 # outer layer

# Cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability (prevents large exponents/extreme values)
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # if we use 1/counts_sum instead, can't get backprop to be exact
probs = counts * counts_sum_inv # softmax turns logits into probabilities
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean() # neg avg of probability of the correct char

# Pytorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(2.2518, grad_fn=<NegBackward0>)

In [40]:
bnbias.shape

torch.Size([1, 200])

In [10]:
# Backward pass (Exercise 1)
# NOTE: sum in forward pass turns into replication/broadcast in backward pass, replication/broadcast in forward pass turns into sum in backward pass over same dimension

# dlogprobs = d/dlogprobs -logprobs[range(n), Yb].mean()
# loss = -(sum prob of correct char)/batch size
# dloss/dprobi = -1/batch size
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n # set grad at index of correct char to be -1/n (all other grads are 0 because they don't influence the loss)

# dprobs = ( d/dprobs log(probs) ) * dlogprobs
dprobs = (1.0 / probs) * dlogprobs # boosts gradient of values with low probability (greater grad for correct/better predicted values)

# dcounts_sum_inv = ( d/dcounts_sum_inv counts*counts_sum_inv ) * dprobs
# counts_sum_inv.shape = 32 x 1 but counts.shape = 32 x 27
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True) # must do sum because counts_sum_inv is replicated across every col of counts
# dcounts = ( d/dcounts counts*counts_sum_inv ) * dprobs
dcounts = counts_sum_inv * dprobs

# dcounts_sum = ( d/dcounts_sum 1/counts_sum ) * dcounts_sum_inv
dcounts_sum = (-counts_sum**-2)*dcounts_sum_inv 

# dcounts = ( d/dcounts count1 + count2 + count3 ... etc) *d_counts_sum
dcounts += torch.ones_like(counts) * dcounts_sum # replicate grad to all counts

# dnorm_logits = ( d/dnorm_logits e^norm_logits ) * dcounts
dnorm_logits = counts * dcounts

# dlogits = ( d/dlogits logits - logit_maxes ) * dnorm_logits
dlogits = dnorm_logits.clone()
# dlogit_maxes = ( d/dlogit_maxes logits - logit_maxes ) * dnorm_logits
# purpose of logit_maxes is to scale down to prevent extreme values when exponentiated
# gradient of logit_maxes (dlogit_maxes) should be ~0 bc it doesn't impact loss
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True) # sum because of replication (preserve dimensions)

# dlogits = ( d/dlogits logits.max(1, keepdim=True).values ) * dlogit_maxes
# scatter dlogit_maxes to correct position in dlogits
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

# dh = ( d/dh h * W2 + b2) * dlogits
dh = dlogits @ W2.T # only way to get correct shape of 32x64

# dW2 = ( d/dW2 h * W2 + b2) * dlogits
dW2 = h.T @ dlogits

# db2 = ( d/db2 h * W2 + b2) * dlogits .sum because of broadcasting
db2 = dlogits.sum(0)

# dhpreact = ( d/dhpreact tanh(hpreact) ) * dh
dhpreact = (1.0 - h**2) * dh

# dbngain = ( d/dbngain bngain * bnraw + bnbias ) * dhpreact
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)

# dbnraw = ( d/dbnraw bngain * bnraw + bnbias ) * dhpreact
dbnraw = bngain * dhpreact

# dbnbias = ( d/dbnbias bngain * bnraw + bnbias ) * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)

# dbndiff = ( d/dbndiff bndiff * bnvar_inv) * dbnraw
dbndiff = bnvar_inv * dbnraw

# dbnvar_inv = ( d/dbnvar_inv bndiff * bnvar_inv) * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

# dbnvar = ( d/dvnvar (bnvar + 1e-5)**-0.5 ) * dbnvar_inv
dbnvar = (-0.5*(bnvar+1e-5)**-1.5) * dbnvar_inv

# dbndiff2 = ( d/dbndiff2 1/(n-1)*(bndiff2).sum(0, keepdim=True) ) * dbnvar
dbndiff2 = torch.ones_like(bndiff2) * (1.0/(n-1)) * dbnvar

# dbndiff = ( d/dbndiff bndiff**2) * dbndiff2
dbndiff += 2 * bndiff * dbndiff2

# dhprebn = ( d/dhprebn hprebn - bnmeani) * bdndiff
dhprebn = dbndiff.clone()

# dbnmeani = ( d/dbnmeani hprebn - bnmeani) * bdndiff
dbnmeani = (-dbndiff).sum(0)

# dhprebn = ( d/dhprebn 1/n*hprebn.sum(0, keepdim=True) ) * dbnmeani
dhprebn += 1.0/n * torch.ones_like(hprebn) * dbnmeani

# dembcat = ( d/dembcat embcat @ W1 + b1 ) * dhprebn
dembcat = dhprebn @ W1.T

# dW1 = ( d/W1 embcat @ W1 + b1 ) * dhprebn
dW1 = embcat.T @ dhprebn

# db1 = ( d/b1 embcat @ W1 + b1 ) * dhprebn
db1 = dhprebn.sum(0)

# demb = ( d/demb embcat = emb.view(emb.shape[0], -1) ) * dembcat
demb = dembcat.view(emb.shape) # embcat rearranged (concatenated) the representation of emb -> rearrange dembcat to match emb gives gradients/backward pass of emb

# dC = ( d/dC C[Xb] ) * demb
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]): # iterate over all elements of Xb
    for j in range (Xb.shape[1]):
        ix = Xb[k, j] # iterates through every index of Xb
        dC[ix] += demb[k, j]

# Check our math
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnraw', dbnraw, bnraw)
cmp('bnbias', dbnbias, bnbias)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff:

In [11]:
# forward pass: emb = C[Xb]
print(emb.shape, C.shape, Xb.shape)
print(Xb[:5])

# emb has 32 examples, 3 chars, each char with 10D embedding
# C = lookup table with 27 chars each with 10D embedding
# Looked up indices of C for examples in Xb

# backward pass -> re-route the gradients back to their respective inputs

torch.Size([32, 3, 10]) torch.Size([27, 10]) torch.Size([32, 3])
tensor([[ 1,  1,  4],
        [18, 14,  1],
        [11,  5,  9],
        [ 0,  0,  1],
        [12, 15, 14]])


In [12]:
# Exercise 2 - simplifying the backward pass because many operations cancel
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff', (loss_fast - loss).item())

3.3277997970581055 diff 0.0


In [13]:
# go from loss to dlogits in one step (dlogits as function of logits and Ybs)

dlogits = F.softmax(logits, 1) # softmax along rows of logits -> turns logits into probabilities
dlogits[range(n), Yb] -= 1 # subtract 1 at correct indices
dlogits /= n # scale down by n because loss is a mean

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 7.2177499532699585e-09


In [14]:
logits.shape, dlogits.shape

(torch.Size([32, 27]), torch.Size([32, 27]))

In [15]:
dlogits[0]*n # gradient scales up probability of correct char and down all other probs

tensor([ 0.0755,  0.0865,  0.0171,  0.0503,  0.0201,  0.0810,  0.0221,  0.0396,
        -0.9816,  0.0342,  0.0377,  0.0380,  0.0353,  0.0278,  0.0356,  0.0136,
         0.0096,  0.0190,  0.0162,  0.0527,  0.0475,  0.0203,  0.0280,  0.0693,
         0.0585,  0.0257,  0.0203], grad_fn=<MulBackward0>)

In [16]:
# Exercise 3 -> single expression for batch norm backward pass from loss to logits

# forward pass
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(9.5367e-07, grad_fn=<MaxBackward1>)


In [17]:
# calculate dhprebn given dhpreact

# hprebn = xi in paper
# hpreact = yi in paper

# dlogits = dLoss/dyi
# we want dLoss/dxi

# sum gradients of dL/dsigma when calculating dL/dmu because many gradients coming into sigma go to one mu
# xi = mu for batch norm which simplifies the term significantly

dhprebn = bngain * bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


In [20]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
with torch.no_grad():

    # kick off optimization
    for i in range(max_steps):
    
        # minibatch construct
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
        Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
    
        # forward pass
        emb = C[Xb] # embed the characters into vectors
        embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
        # Linear layer
        hprebn = embcat @ W1 + b1 # hidden layer pre-activation
        # BatchNorm layer
        # -------------------------------------------------------------
        bnmean = hprebn.mean(0, keepdim=True)
        bnvar = hprebn.var(0, keepdim=True, unbiased=True)
        bnvar_inv = (bnvar + 1e-5)**-0.5
        bnraw = (hprebn - bnmean) * bnvar_inv
        hpreact = bngain * bnraw + bnbias
        # -------------------------------------------------------------
        # Non-linearity
        h = torch.tanh(hpreact) # hidden layer
        logits = h @ W2 + b2 # output layer
        loss = F.cross_entropy(logits, Yb) # loss function
    
        # backward pass
        for p in parameters:
          p.grad = None
        #loss.backward() # use this for correctness comparisons, delete it later!
    
        # manual backprop! #swole_doge_meme
        # -----------------
        dlogits = F.softmax(logits, 1) # softmax along rows of logits -> turns logits into probabilities
        dlogits[range(n), Yb] -= 1 # subtract 1 at correct indices
        dlogits /= n # scale down by n because loss is a mean
        # 2nd layer
        dh = dlogits @ W2.T # only way to get correct shape of 32x64
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)
        # tanh
        dhpreact = (1.0 - h**2) * dh
        # batchnorm backprop
        dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
        dbnbias = dhpreact.sum(0, keepdim=True)
        dhprebn = bngain * bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
        # 1st layer
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(0)
        # embedding
        demb = dembcat.view(emb.shape) # embcat rearranged (concatenated) the representation of emb -> rearrange dembcat to match emb gives gradients/backward pass of emb
        dC = torch.zeros_like(C)
        for k in range(Xb.shape[0]): # iterate over all elements of Xb
            for j in range (Xb.shape[1]):
                ix = Xb[k, j] # iterates through every index of Xb
                dC[ix] += demb[k, j]
        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
        # -----------------
        
        # update
        lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
        for p, grad in zip(parameters, grads):
          #p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
          p.data += -lr * grad # new way of swole doge TODO: enable
    
        # track stats
        if i % 10000 == 0: # print every once in a while
          print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
        lossi.append(loss.log10().item())
    
        #if i >= 100: # TODO: delete early breaking when you're ready to train the full net
            #break

12297
      0/ 200000: 3.8062
  10000/ 200000: 2.2377
  20000/ 200000: 2.3784
  30000/ 200000: 2.4491
  40000/ 200000: 1.9996
  50000/ 200000: 2.3794
  60000/ 200000: 2.3839
  70000/ 200000: 2.1052
  80000/ 200000: 2.3578
  90000/ 200000: 2.1218
 100000/ 200000: 1.9627
 110000/ 200000: 2.3653
 120000/ 200000: 2.0218
 130000/ 200000: 2.3703
 140000/ 200000: 2.3065
 150000/ 200000: 2.1820
 160000/ 200000: 2.0074
 170000/ 200000: 1.7847
 180000/ 200000: 2.0074
 190000/ 200000: 1.8957


In [19]:
# check our gradients

#for p,g in zip(parameters, grads):
    #cmp(str(tuple(p.shape)), g, p)

(27, 10)        | exact: False | approximate: True  | maxdiff: 1.1641532182693481e-08
(30, 200)       | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
(200,)          | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09
(200, 27)       | exact: False | approximate: True  | maxdiff: 1.862645149230957e-08
(27,)           | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09
(1, 200)        | exact: False | approximate: True  | maxdiff: 2.7939677238464355e-09
(1, 200)        | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09


In [21]:
# evaluate the batch norm (because we didn't keep track of the mean)

with torch.no_grad():
    # pass the training set through
    emb = C[Xtr]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    # measure the mean/stdev over entire training set
    bnmean = hpreact.mean(0, keepdim=True)
    bnvar = hpreact.var(0, keepdim=True)

In [22]:
# evaluate train and val loss

@torch.no_grad() # disables gradient tracking
def split_loss(split):
    x,y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x] # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
    h = torch.tanh(hpreact) # (N, n_hidden)
    logits = h @ W2 + b2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.070390462875366
val 2.110736608505249


In [23]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # ------------
      # forward pass:
      # Embedding
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # ------------
      # Sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carmah.
amille.
khi.
mri.
reity.
salaysie.
mahnen.
delynn.
jareei.
ner.
kiah.
maiivon.
leigh.
ham.
joce.
quint.
shoine.
liven.
coraelo.
dearyni.
